In [2]:
# %% Chargement
import pandas as pd

pd.set_option("display.max_rows", 100)
df = pd.read_parquet("../data/offline/offline.parquet")
print(df.shape)

# %% Types de colonnes
print(df.dtypes)

# %% Valeurs manquantes (%)
missing = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
print(missing[missing > 0])

# %% Taille des recherches (hôtels par srch_id)
sizes = df.groupby("srch_id").size()
print(sizes.describe())

# %% Position vs clic / réservation
print(df.groupby("position")[["click_bool", "booking_bool"]].mean().head(40))

# %% Effet de random_bool (ordre aléatoire vs ordre Expedia)
print(df.groupby("random_bool")[["click_bool", "booking_bool"]].mean())
print(df["random_bool"].value_counts(normalize=True))

(5751004, 54)
srch_id                                 int64
date_time                      datetime64[us]
site_id                                 int64
visitor_location_country_id             int64
visitor_hist_starrating               float64
visitor_hist_adr_usd                  float64
prop_country_id                         int64
prop_id                                 int64
prop_starrating                         int64
prop_review_score                     float64
prop_brand_bool                         int64
prop_location_score1                  float64
prop_location_score2                  float64
prop_log_historical_price             float64
position                                int64
price_usd                             float64
promotion_flag                          int64
srch_destination_id                     int64
srch_length_of_stay                     int64
srch_booking_window                     int64
srch_adults_count                       int64
srch_children_count 

In [2]:
import os
print(os.getcwd())                          # où tu es
print(os.path.exists("data/offline/offline.parquet"))  # False = fichier absent

c:\Users\LENOVO\Desktop\expedia-ranking\notebooks
False


In [4]:
# Valeurs manquantes (%)
missing = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
print(missing[missing > 0].to_string())

comp1_rate_percent_diff      98.71
comp6_rate_percent_diff      98.35
comp1_rate                   98.30
comp1_inv                    98.25
comp4_rate_percent_diff      97.31
gross_bookings_usd           97.25
comp7_rate_percent_diff      97.16
comp6_rate                   95.92
comp6_inv                    95.59
visitor_hist_starrating      95.05
visitor_hist_adr_usd         95.02
srch_query_affinity_score    93.69
comp7_rate                   93.58
comp4_rate                   93.54
comp7_inv                    93.02
comp4_inv                    92.66
comp3_rate_percent_diff      89.99
comp2_rate_percent_diff      88.56
comp8_rate_percent_diff      86.98
comp5_rate_percent_diff      82.47
comp3_rate                   68.05
comp3_inv                    65.47
comp8_rate                   60.29
comp8_inv                    58.89
comp2_rate                   58.80
comp2_inv                    56.60
comp5_rate                   55.75
comp5_inv                    53.17
orig_destination_dis

In [5]:
# Position vs clic / réservation
print(df.groupby("position")[["click_bool", "booking_bool"]].mean().head(40).to_string())

          click_bool  booking_bool
position                          
1           0.193739      0.141740
2           0.134592      0.094480
3           0.105273      0.071335
4           0.087725      0.057416
5           0.008660      0.000000
6           0.071108      0.045095
7           0.059950      0.037196
8           0.052597      0.032025
9           0.046355      0.027078
10          0.043321      0.025018
11          0.003383      0.000000
12          0.037139      0.021405
13          0.033619      0.018600
14          0.030601      0.016631
15          0.029055      0.015692
16          0.028210      0.014756
17          0.003084      0.000000
18          0.024392      0.012618
19          0.022862      0.011744
20          0.022130      0.010964
21          0.020548      0.010105
22          0.020005      0.009333
23          0.002146      0.000000
24          0.018153      0.008930
25          0.016119      0.007845
26          0.016159      0.007776
27          0.015588

In [3]:
print(df["position"].value_counts().sort_index().head(30).to_string())
print(df.groupby(["random_bool", "position"])["click_bool"].mean().unstack(0).head(25).to_string())

position
1     229159
2     229107
3     229129
4     222149
5       8199
6     229005
7     224003
8     218985
9     214344
10    199858
11     10642
12    205608
13    201612
14    197578
15    193665
16    178978
17     11026
18    186002
19    182225
20    178488
21    174758
22    160407
23     10720
24    167301
25    163411
26    159722
27    156083
28    152426
29    148553
30    144346
random_bool         0         1
position                       
1            0.217286  0.140306
2            0.147153  0.106070
3            0.111453  0.091245
4            0.092429  0.077454
5            0.008647  0.008955
6            0.071709  0.069744
7            0.059555  0.060850
8            0.051638  0.054790
9            0.044196  0.051309
10           0.041601  0.047035
11           0.003538  0.000000
12           0.034414  0.043418
13           0.030467  0.040913
14           0.027217  0.038447
15           0.025973  0.036219
16           0.025051  0.035070
17           0.003041  0.

In [4]:
print(df.groupby("srch_id").size().describe())

count    229645.000000
mean         25.043019
std           9.082159
min           5.000000
25%          18.000000
50%          30.000000
75%          32.000000
max          36.000000
dtype: float64


In [5]:
g = df.groupby("srch_id")[["click_bool", "booking_bool"]].max()
print(g.mean())

click_bool      1.000000
booking_bool    0.689042
dtype: float64


In [6]:
print(df["price_usd"].describe(percentiles=[.01, .5, .99, .999]).to_string())
df["month"] = df["date_time"].dt.to_period("M")
print(df.groupby("month").agg(price=("price_usd", "median"),
                              click=("click_bool", "mean"),
                              booking=("booking_bool", "mean")))
df["relevance"] = df["booking_bool"] * 5
df.loc[(df["click_bool"] == 1) & (df["booking_bool"] == 0), "relevance"] = 1
num = df.select_dtypes("number").drop(columns=["click_bool", "booking_bool", "gross_bookings_usd", "position"])
print(num.corrwith(df["relevance"]).sort_values().to_string())

count    5.751004e+06
mean     2.200247e+02
std      9.869595e+03
min      0.000000e+00
1%       3.300000e+01
50%      1.192000e+02
99%      6.050000e+02
99.9%    2.199897e+03
max      9.661340e+06
          price     click   booking
month                              
2012-11  117.00  0.044722  0.028274
2012-12  115.00  0.044267  0.027902
2013-01  121.82  0.043663  0.026296
2013-02  122.78  0.044249  0.027660
2013-03  122.00  0.044865  0.027654
random_bool                   -0.068470
srch_length_of_stay           -0.018484
srch_booking_window           -0.015291
comp6_inv                     -0.007839
comp3_inv                     -0.006691
comp7_inv                     -0.006654
comp4_inv                     -0.004168
srch_adults_count             -0.003971
comp5_inv                     -0.002999
orig_destination_distance     -0.002407
comp8_rate_percent_diff       -0.001748
prop_location_score1          -0.001096
comp5_rate_percent_diff       -0.000959
comp4_rate_percent_diff       